In [3]:
import pandas as pd
import numpy as np
import torch
import os
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

2026-02-06 14:40:56.755026: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770388856.936162      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770388856.988033      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770388857.426117      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770388857.426149      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770388857.426152      55 computation_placer.cc:177] computation placer alr

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"
DATA_DIR = "/kaggle/input/prompt-recovery1" # Updated path per your instruction

# Style Mapping
LABEL2ID = {
    "Formal": 0, "Informal": 1, "Optimistic": 2, "Pessimistic": 3,
    "Humorous": 4, "Serious": 5, "Inspiring": 6, "Authoritative": 7, "Persuasive": 8
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [5]:
def load_clean_data(file_name, is_test=False):
    path = os.path.join(DATA_DIR, file_name)
    df = pd.read_csv(path)
    
    # Standardize column names (strips spaces and converts to upper for matching)
    df.columns = df.columns.str.strip().str.upper()
    
    INPUT_COL = "CHANGE STYLE"
    LABEL_COL = "STYLE"
    ID_COL = "ID"
    
    # Cleaning
    df = df.dropna(subset=[INPUT_COL])
    df[INPUT_COL] = df[INPUT_COL].astype(str).str.strip()
    
    if not is_test:
        df = df.dropna(subset=[LABEL_COL])
        df['label'] = df[LABEL_COL].str.strip().map(LABEL2ID)
        df = df.dropna(subset=['label'])
        df['label'] = df['label'].astype(int)
        return df[[ID_COL, INPUT_COL, 'label']]
    else:
        return df[[ID_COL, INPUT_COL]]

print("Loading Train, Dev, and Test sets...")
train_df = load_clean_data("train.csv")
dev_df = load_clean_data("dev.csv")
test_df = load_clean_data("test.csv", is_test=True)

Loading Train, Dev, and Test sets...


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TeluguStyleDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.encodings = tokenizer(
            df["CHANGE STYLE"].tolist(), 
            truncation=True, 
            padding=True, 
            max_length=512
        )
        self.is_test = is_test
        if not is_test:
            self.labels = df["label"].tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if not self.is_test:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings.input_ids)

train_dataset = TeluguStyleDataset(train_df)
dev_dataset = TeluguStyleDataset(dev_df)
test_dataset = TeluguStyleDataset(test_df, is_test=True)

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "macro_f1": f1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=9,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(DEVICE)

training_args = TrainingArguments(
    output_dir="./telugu_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    
    # --- DISK FIXES ---
    save_total_limit=1,              # ONLY keep the 1 best model. Deletes others.
    load_best_model_at_end=True,     # Required for high accuracy
    metric_for_best_model="macro_f1",
    
    # --- ACCURACY BOOSTERS ---
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    num_train_epochs=15,             # Increased epochs
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    overwrite_output_dir=True,       # Overwrites old files if they exist
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/IndicBERTv2-MLM-only and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
print("\n🚀 Starting Fine-Tuning on IndicBERT v2...")
trainer.train()


🚀 Starting Fine-Tuning on IndicBERT v2...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,2.198831,0.114094,0.044088
2,No log,2.009138,0.258389,0.230516
3,No log,1.776000,0.355705,0.327684
4,No log,1.725969,0.348993,0.317687
5,No log,1.665874,0.359060,0.365904
6,1.817000,1.655971,0.392617,0.393262
7,1.817000,1.683692,0.372483,0.369104
8,1.817000,1.731295,0.365772,0.368324
9,1.817000,1.777331,0.385906,0.383955


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

TrainOutput(global_step=846, training_loss=1.537524428491615, metrics={'train_runtime': 1627.3873, 'train_samples_per_second': 27.523, 'train_steps_per_second': 0.866, 'total_flos': 7071290905688064.0, 'train_loss': 1.537524428491615, 'epoch': 9.0})

In [9]:
print("\n🔍 Running predictions on Test Set...")
test_results = trainer.predict(test_dataset)
test_preds = np.argmax(test_results.predictions, axis=-1)


🔍 Running predictions on Test Set...


In [10]:
test_df['STYLE'] = [ID2LABEL[p] for p in test_preds]
output_file = "submission.csv"
test_df[['ID', 'STYLE']].to_csv(output_file, index=False)

print(f"\n✅ Success! File saved as: {output_file}")
print(test_df[['ID', 'STYLE']].head())


✅ Success! File saved as: submission.csv
              ID       STYLE
0  PR_TE_TE_0001      Formal
1  PR_TE_TE_0002      Formal
2  PR_TE_TE_0003   Inspiring
3  PR_TE_TE_0004     Serious
4  PR_TE_TE_0005  Optimistic
